# Logistics Data Warehouse — ETL Notebook
**Course:** Data Warehousing and Visualisation  
**Author:** Soelema Remenda Pieka  
**University:** Università della Calabria  
**Academic Year:** 2025/2026  

---

This notebook implements the full ETL pipeline for the logistics data warehouse 
designed in Phase 1. The star schema, all dimension and fact table definitions, 
and all design decisions (attribute tree editing steps, derived measures) are 
documented in the project report.

**Dataset:** Synthetic Logistics Operations Database (2022–2024)  
**Source:** https://www.kaggle.com/datasets/yogape/logistics-operations-database

---
## 0. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import os

# Paths
RAW_PATH = 'data/raw/'
OUTPUT_PATH = 'data/output/'

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Setup complete")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Setup complete
Pandas version: 2.2.2
NumPy version: 1.26.4


---
# EXTRACT
---

## 1. Load Raw CSVs
*Phase 1 reference: Section 1.1 — Data Source Description*  
Load all 10 selected source files into pandas DataFrames.

In [2]:
# Load all 10 source CSV files
drivers = pd.read_csv(RAW_PATH + 'drivers.csv')
trucks = pd.read_csv(RAW_PATH + 'trucks.csv')
trailers = pd.read_csv(RAW_PATH + 'trailers.csv')
customers = pd.read_csv(RAW_PATH + 'customers.csv')
facilities = pd.read_csv(RAW_PATH + 'facilities.csv')
routes = pd.read_csv(RAW_PATH + 'routes.csv')
loads = pd.read_csv(RAW_PATH + 'loads.csv')
trips = pd.read_csv(RAW_PATH + 'trips.csv')
delivery_events = pd.read_csv(RAW_PATH + 'delivery_events.csv')
fuel_purchases = pd.read_csv(RAW_PATH + 'fuel_purchases.csv')

print("All files loaded successfully")
print(f"\nRow counts:")
print(f"  drivers:         {len(drivers):>10,}")
print(f"  trucks:          {len(trucks):>10,}")
print(f"  trailers:        {len(trailers):>10,}")
print(f"  customers:       {len(customers):>10,}")
print(f"  facilities:      {len(facilities):>10,}")
print(f"  routes:          {len(routes):>10,}")
print(f"  loads:           {len(loads):>10,}")
print(f"  trips:           {len(trips):>10,}")
print(f"  delivery_events: {len(delivery_events):>10,}")
print(f"  fuel_purchases:  {len(fuel_purchases):>10,}")

All files loaded successfully

Row counts:
  drivers:                150
  trucks:                 120
  trailers:               180
  customers:              200
  facilities:              50
  routes:                  58
  loads:               85,410
  trips:               85,410
  delivery_events:    170,820
  fuel_purchases:     196,442


In [3]:
# Dictionary of all dataframes for easy iteration
dataframes = {
    'drivers': drivers,
    'trucks': trucks,
    'trailers': trailers,
    'customers': customers,
    'facilities': facilities,
    'routes': routes,
    'loads': loads,
    'trips': trips,
    'delivery_events': delivery_events,
    'fuel_purchases': fuel_purchases
}

# Print shape and column names for each dataframe
for name, df in dataframes.items():
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Dtypes:\n{df.dtypes}")


  DRIVERS
  Shape: (150, 12)
  Columns: ['driver_id', 'first_name', 'last_name', 'hire_date', 'termination_date', 'license_number', 'license_state', 'date_of_birth', 'home_terminal', 'employment_status', 'cdl_class', 'years_experience']
  Dtypes:
driver_id            object
first_name           object
last_name            object
hire_date            object
termination_date     object
license_number       object
license_state        object
date_of_birth        object
home_terminal        object
employment_status    object
cdl_class            object
years_experience      int64
dtype: object

  TRUCKS
  Shape: (120, 11)
  Columns: ['truck_id', 'unit_number', 'make', 'model_year', 'vin', 'acquisition_date', 'acquisition_mileage', 'fuel_type', 'tank_capacity_gallons', 'status', 'home_terminal']
  Dtypes:
truck_id                 object
unit_number               int64
make                     object
model_year                int64
vin                      object
acquisition_date         ob

---
# TRANSFORM
---

## 2. Data Inspection
*Phase 1 reference: Section 1.2 — Reconciled Database Schema*  
Verify that the raw data matches the E/R schema designed in Phase 1.

In [4]:
# Quick inspection summary for all dataframes
# Shows shape, missing values, duplicates and basic stats for each table

for name, df in dataframes.items():
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(f"  Shape:            {df.shape}")
    print(f"  Missing values:   {df.isnull().sum().sum()}")
    print(f"  Duplicate rows:   {df.duplicated().sum()}")
    print(f"  Duplicate IDs:    {df.duplicated(subset=df.columns[0]).sum()}")


  DRIVERS
  Shape:            (150, 12)
  Missing values:   124
  Duplicate rows:   0
  Duplicate IDs:    0

  TRUCKS
  Shape:            (120, 11)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  TRAILERS
  Shape:            (180, 9)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  CUSTOMERS
  Shape:            (200, 8)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  FACILITIES
  Shape:            (50, 9)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  ROUTES
  Shape:            (58, 9)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  LOADS
  Shape:            (85410, 12)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  TRIPS
  Shape:            (85410, 12)
  Missing values:   5066
  Duplicate rows:   0
  Duplicate IDs:    0

  DELIVERY_EVENTS
  Shape:            (170820, 11)
  Missing values:   0
  Duplicate rows:   0
  Duplicate IDs:    0

  FUEL_PURCHASES
  Sha

In [5]:
# Detailed missing value inspection for tables with nulls

for name in ['drivers', 'trips', 'fuel_purchases']:
    df = dataframes[name]
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"\n{'='*50}")
    print(f"  {name.upper()} — Missing Values")
    print(f"{'='*50}")
    print(missing)
    print(f"  Total missing: {missing.sum()} / {len(df) * len(df.columns)} cells")


  DRIVERS — Missing Values
termination_date    124
dtype: int64
  Total missing: 124 / 1800 cells

  TRIPS — Missing Values
driver_id     1714
truck_id      1672
trailer_id    1680
dtype: int64
  Total missing: 5066 / 1024920 cells

  FUEL_PURCHASES — Missing Values
truck_id     3880
driver_id    3988
dtype: int64
  Total missing: 7868 / 2160862 cells


In [6]:
# Check if missing driver, truck, trailer overlap on the same trips
missing_trips = trips[trips['driver_id'].isnull() | 
                      trips['truck_id'].isnull() | 
                      trips['trailer_id'].isnull()]

print(f"Trips missing at least one assignment: {len(missing_trips)}")
print(f"\nMissing combinations:")
print(missing_trips[['driver_id','truck_id','trailer_id']].isnull().sum())
print(f"\nTrips missing all three: {missing_trips[['driver_id','truck_id','trailer_id']].isnull().all(axis=1).sum()}")

Trips missing at least one assignment: 4952

Missing combinations:
driver_id     1714
truck_id      1672
trailer_id    1680
dtype: int64

Trips missing all three: 0


In [7]:
# Check if trips with missing assignments still have financial data via loads
missing_trips = trips[trips['driver_id'].isnull() | 
                      trips['truck_id'].isnull() | 
                      trips['trailer_id'].isnull()]

# Check their load_ids are valid
valid_loads = missing_trips['load_id'].isin(loads['load_id']).sum()
print(f"Missing assignment trips with valid load_id: {valid_loads} / {len(missing_trips)}")

Missing assignment trips with valid load_id: 4952 / 4952


In [8]:
# Check fuel purchases with missing truck_id or driver_id
missing_fuel = fuel_purchases[fuel_purchases['truck_id'].isnull() | 
                              fuel_purchases['driver_id'].isnull()]

print(f"Fuel purchases missing truck_id or driver_id: {len(missing_fuel)}")

# Check if they still have valid trip_id for aggregation
valid_trips = missing_fuel['trip_id'].isin(trips['trip_id']).sum()
print(f"Of those, with valid trip_id: {valid_trips} / {len(missing_fuel)}")

# Check if gallons and total_cost are intact
print(f"\nMissing gallons in affected rows: {missing_fuel['gallons'].isnull().sum()}")
print(f"Missing total_cost in affected rows: {missing_fuel['total_cost'].isnull().sum()}")

Fuel purchases missing truck_id or driver_id: 7774
Of those, with valid trip_id: 7774 / 7774

Missing gallons in affected rows: 0
Missing total_cost in affected rows: 0


In [9]:
# Check 1 — trip_status values
print("Trip status values:")
print(trips['trip_status'].value_counts())

# Check 2 — state to city hypothesis in routes
print("\nOrigin states with more than 1 city:")
print(routes.groupby('origin_state')['origin_city'].nunique().sort_values(ascending=False).head(10))

print("\nDestination states with more than 1 city:")
print(routes.groupby('destination_state')['destination_city'].nunique().sort_values(ascending=False).head(10))

Trip status values:
trip_status
Completed    85410
Name: count, dtype: int64

Origin states with more than 1 city:
origin_state
TX    2
AZ    1
NV    1
TN    1
PA    1
OR    1
OH    1
NY    1
NC    1
CO    1
Name: origin_city, dtype: int64

Destination states with more than 1 city:
destination_state
TX    2
CA    1
CO    1
TN    1
PA    1
OR    1
OH    1
NY    1
NV    1
NC    1
Name: destination_city, dtype: int64


In [10]:
# Check Texas cities in routes
print("Texas origin cities:")
print(routes[routes['origin_state'] == 'TX']['origin_city'].unique())

print("\nTexas destination cities:")
print(routes[routes['destination_state'] == 'TX']['destination_city'].unique())

Texas origin cities:
['Dallas' 'Houston']

Texas destination cities:
['Dallas' 'Houston']


## 3. Data Quality Assessment (ISO 25012)
*Phase 1 reference: Section 2.2 — Data Quality Assessment*

### 3.1 Completeness
### 3.2 Accuracy
### 3.3 Consistency
### 3.4 Uniqueness
### 3.5 Timeliness

## 4. Data Cleaning Pipeline
### 4.1 Missing Values
### 4.2 Outliers
### 4.3 Standardization

## 5. Build Dimension Tables
*Phase 1 reference: Section 1.4 — Star Schema, attribute tree editing Step 1 (pruning)*  
Column selection follows the pruning decisions documented in the attribute tree editing steps.

## 6. Build Fact Table
*Phase 1 reference: Section 1.4 — fact\_trips definition and glossary of measures*

### 6.1 Base joins
### 6.2 Pivot delivery\_events
*Phase 1 reference: Attribute tree editing Step 2 — pivoting and grafting of delivery\_events*

### 6.3 Aggregate fuel\_purchases
*Phase 1 reference: Attribute tree editing Step 4 — aggregation and grafting of fuel\_purchases*

### 6.4 Add dim\_date
*Phase 1 reference: Attribute tree editing Step 6 — promoting dispatch\_date to dim\_date*

### 6.5 Derived columns
*Phase 1 reference: Section 1.5 — Glossary of measures (derived measures section)*

---
# LOAD
---
## 7. Export to CSV
Export all final dimension and fact tables to `data/output/`.

---
## 8. Validation & Sanity Checks
Row count verification, referential integrity checks, and KPI spot checks.